In [1]:
model_name = "vit-ragdoll"
model_file = "vit.mlir.v1"

import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

image = torch.randn(1, 3, 224, 224)
image_np = image.detach().cpu().numpy()
image_transposed = torch.randn(1, 224, 224, 3)
image_np_transposed = image.detach().cpu().numpy()
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
output = model(image)
grad = torch.randn_like(output)
grad_np = grad.cpu().numpy()

BENCHMARK_REPEAT=33
df = pd.DataFrame()

In [2]:

#!iree-compile recompute.mlir \
!iree-compile {model_file}.recompute \
-o {model_file}.recompute.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

!iree-compile {model_file}.storeall \
-o {model_file}.storeall.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

"""
!iree-compile {model_file}.hybrid \
-o {model_file}.hybrid.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86
"""

!iree-compile {model_file}.heuristic \
-o {model_file}.heuristic.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

#!iree-compile {model_file}.heuristic{"-mulswap"} \
#-o {model_file}.heuristic{"-mulswap"}.vmfb \
#--iree-hal-target-backends=cuda \
#--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
#--iree-hal-cuda-llvm-target-arch=sm_86

recompute = model_file+".recompute.vmfb"
storeall = model_file+".storeall.vmfb"
#hybrid = model_file+".hybrid.vmfb"
heuristic = model_file+".heuristic.vmfb"
#heuristic2 = model_file+".heuristic-mulswap.vmfb"

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

#recompute_fb, storeall_fb = [
#    load_executable(x) for x in [recompute, storeall]
#]
recompute_fb = load_executable(recompute)
storeall_fb = load_executable(storeall)
#hybrid_fb = load_executable(hybrid)
heuristic_fb = load_executable(heuristic)
#heuristic2_fb = load_executable(heuristic2)

<unknown>:0: error: LLVM Translation failed for operation: builtin.unrealized_conversion_cast
<unknown>:0: note: see current operation: %67 = "builtin.unrealized_conversion_cast"(%66) : (!llvm.array<1 x array<1 x vector<4xf32>>>) -> vector<1x1x4xf32>
vit.mlir.v1.storeall:1376:11: error: failed to translate the MLIR LLVM dialect to the native llvm::Module
    %43 = tosa.add %41, %42 : (tensor<1x197x3072xf32>, tensor<1x197x3072xf32>) -> tensor<1x197x3072xf32>
          ^
vit.mlir.v1.storeall:1115:3: note: called from
  func.func @dforward(%arg0: tensor<1x1000xf32>) -> tensor<1x3x224x224xf32> {
  ^
vit.mlir.v1.storeall:1376:11: note: see current operation: 
"hal.executable.variant"() ({
  "hal.executable.export"() ({
  ^bb0(%arg0: !hal.device):
    %0 = "arith.constant"() <{value = 24 : index}> : () -> index
    %1 = "arith.constant"() <{value = 197 : index}> : () -> index
    %2 = "arith.constant"() <{value = 1 : index}> : () -> index
    "hal.return"(%0, %1, %2) : (index, index, index) 

FileNotFoundError: [Errno 2] No such file or directory: 'vit.mlir.v1.storeall.vmfb'

In [ ]:
#f1 = timeit("recompute_fb.forward(image_np)") / BENCHMARK_REPEAT
f1 = timeit("recompute_fb.forward(image_np_transposed)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("recompute_fb.dforward(grad_np)") /  BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-backward in timeit: ', b1)
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-Recompute")])
print(df)

In [ ]:
f1 = timeit("storeall_fb.forward(image_np_transposed)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("storeall_fb.dforward(grad_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', b1)
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-Storeall")])
print(df)

In [ ]:
f1 = timeit("heuristic_fb.forward(image_np_transposed)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("heuristic_fb.dforward(grad_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', b1)
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-Heuristic")])
print(df)

In [ ]:
df.style.hide(axis="index")
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")